## This notebook implements a complete PySpark pipeline for the IDS2018 dataset, covering data ingestion, preprocessing, feature selection, model training, threshold tuning, and metrics reporting.  
## The focus of this run is to evaluate performance when deploying a single Spark worker configured with four CPU cores, in order to benchmark accuracy, training time, and parallelism efficiency under limited computational resources.


# Importing Essential Libraries

In [1]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark import StorageLevel

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

import time
from pathlib import Path


## PySpark Session Setup with Single Worker (4 Cores)

In [2]:
import os, sys
from pyspark.sql import SparkSession

# Stop any old Spark session 
try:
    spark.stop()
except:
    pass

MASTER_URL = "spark://127.0.0.1:7077"
CORES = 4  

# Use the same Python for both driver and executors
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder
         .master(MASTER_URL)
         .appName(f"IDS2018-Notebook-{CORES}c-1e")
         .config("spark.executor.instances", "1")
         .config("spark.executor.cores", str(CORES))
         .config("spark.cores.max",      str(CORES))
         .config("spark.executor.memory","6g")
         .config("spark.driver.memory",  "4g")
         # Partitioning tuned for 4 cores
         .config("spark.sql.shuffle.partitions", str(6 * CORES))  # 24
         .config("spark.default.parallelism",    str(3 * CORES))  # 12
         # Network/serialization settings
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         # AQE to reduce shuffle when possible
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
         .config("spark.sql.adaptive.coalescePartitions.minPartitionNum", str(4 * CORES))  # 16
         .config("spark.executorEnv.PYSPARK_PYTHON", sys.executable)
         .getOrCreate())

print("Spark:", spark.version, "| master:", spark.sparkContext.master)
print("cores.max:", spark.sparkContext.getConf().get("spark.cores.max"))
print("executor.cores:", spark.sparkContext.getConf().get("spark.executor.cores"))
print("defaultParallelism:", spark.sparkContext.defaultParallelism)
print("shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))


Spark: 3.5.6 | master: spark://127.0.0.1:7077
cores.max: 4
executor.cores: 4
defaultParallelism: 12
shuffle.partitions: 24


## IDS2018 Data Preprocessing: Ingest, Clean, and Label

In [3]:
import os
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

CSV_DIR = r"C:\Users\HP\bigdata\sparkk2\data"
OUT_DIR = r"C:\Users\HP\bigdata\sparkk2\reports"
os.makedirs(OUT_DIR, exist_ok=True)

raw = (spark.read
       .option("header", True)
       .option("inferSchema", False)  
       .option("multiLine", True)
       .option("escape", '"')
       .csv(str(Path(CSV_DIR) / "*.csv")))
raw = raw.dropna(how="all").toDF(*[c.strip() for c in raw.columns])

# Clean and derive binary label
if "Label" in raw.columns:
    df = raw.filter(F.col("Label") != "Label")
    df = df.withColumn("Label_clean", F.lower(F.trim(F.col("Label"))))
else:
    df = raw.withColumn("Label_clean", F.lit(None).cast("string"))

df = df.withColumn(
    "label",
    F.when(F.col("Label_clean") == F.lit("benign"), F.lit(0)).otherwise(F.lit(1)).cast("double")
)

# Drop columns that may leak information
drop_cols = [c for c in ["Flow ID","Timestamp","Src IP","Dst IP","Src Port","Dst Port","Label","Label_clean"] if c in df.columns]
df = df.drop(*drop_cols)

# Safe cast to double
feat_cols = [c for c in df.columns if c != "label"]
def to_double_safe(cname: str):
    return (
        F.when(F.col(cname).isin("", "NaN", "nan", "Infinity", "-Infinity"), None)
         .otherwise(F.col(cname))
         .cast(DoubleType())
         .alias(cname)
    )

df = df.select(*[to_double_safe(c) for c in feat_cols], "label")

# Clean NaN
num_cols = [c for c, t in df.dtypes if c != "label" and t in ("double","int","bigint","float")]
if num_cols:
    df = df.fillna(0.0, subset=num_cols)
    for c in num_cols:
        df = df.withColumn(c, F.when(~F.isnan(F.col(c)), F.col(c)).otherwise(F.lit(0.0)))

# Repartition according to cores and enable caching
target_parts = int(spark.conf.get("spark.sql.shuffle.partitions"))
df = df.repartition(target_parts).persist()

print("Rows:", df.count(), "| Numeric columns:", len(num_cols))
df.groupBy("label").count().orderBy("label").show(truncate=False)


Rows: 3145724 | Numeric columns: 77
+-----+-------+
|label|count  |
+-----+-------+
|0.0  |2162611|
|1.0  |983113 |
+-----+-------+



## Feature selection via RF importance (TRAIN ONLY)

In [4]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark import StorageLevel
from pyspark.sql import functions as F
import pandas as pd, os

SEED = 42
TOPK = 40
SAMPLE_ROWS = 300_000  # speed up importance estimation

# Split train/test if not already available
if "trainDF" not in globals() or "testDF" not in globals():
    trainDF, testDF = df.randomSplit([0.8, 0.2], seed=SEED)
    trainDF = trainDF.persist(StorageLevel.MEMORY_AND_DISK)
    testDF  = testDF.persist(StorageLevel.MEMORY_AND_DISK)

# Assemble all numeric features on TRAIN only
assembler_all = VectorAssembler(inputCols=num_cols, outputCol="features_all")
train_all = (assembler_all
             .transform(trainDF.select(*num_cols, "label"))
             .select("features_all", "label")
             .persist(StorageLevel.MEMORY_AND_DISK))

cnt = train_all.count()
if cnt == 0:
    raise RuntimeError("Empty training set; cannot compute feature importances.")
frac = min(1.0, SAMPLE_ROWS / float(cnt))
train_s = train_all.sample(False, frac, seed=SEED) if frac < 1.0 else train_all

# Small RF to estimate feature importance
rf_tmp = RandomForestClassifier(featuresCol="features_all", labelCol="label",
                                numTrees=60, maxDepth=10, seed=SEED)
rf_tmp_m = rf_tmp.fit(train_s)

# Rank features
imp = pd.Series(rf_tmp_m.featureImportances.toArray(), index=num_cols).sort_values(ascending=False)
TOPK = min(TOPK, len(num_cols))
top_features = list(imp.head(TOPK).index)

print(f"Selected Top-{TOPK} features (from TRAIN only):", len(top_features))
print(top_features[:20])

# Save importance file
imp_df = imp.reset_index(); imp_df.columns = ["feature", "importance"]
imp_path = os.path.join(OUT_DIR, f"feature_importances_train_top{TOPK}.csv")
imp_df.to_csv(imp_path, index=False, encoding="utf-8-sig")
print("Saved importances ->", imp_path)

# Cleanup
train_all.unpersist()
if frac < 1.0:
    train_s.unpersist()


Selected Top-40 features (from TRAIN only): 40
['Fwd Seg Size Min', 'Fwd Pkt Len Max', 'Pkt Len Max', 'Fwd Header Len', 'Init Fwd Win Byts', 'Pkt Len Mean', 'Fwd Pkts/s', 'Flow Pkts/s', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Flow IAT Max', 'Fwd IAT Max', 'Subflow Fwd Byts', 'Pkt Len Var', 'Flow Duration', 'Fwd IAT Tot', 'Bwd Pkts/s', 'TotLen Fwd Pkts', 'Fwd IAT Min', 'Fwd IAT Mean']
Saved importances -> C:\Users\HP\bigdata\sparkk2\reports\feature_importances_train_top40.csv


## Assemble selected features, split, cache, undersample 

In [5]:
from pyspark.ml.feature import VectorAssembler
from pyspark import StorageLevel
from pyspark.sql import functions as F

# Assemble only the selected features to reduce I/O
assembler = VectorAssembler(inputCols=top_features, outputCol="features")
data = (assembler
        .transform(df.select(*(top_features + ["label"])))
        .select("features", F.col("label").cast("double").alias("label")))

# Repartition
target_parts = int(spark.conf.get("spark.sql.shuffle.partitions"))
data = data.repartition(target_parts)

# Split
trainDF, testDF = data.randomSplit([0.8, 0.2], seed=42)
trainDF = trainDF.persist(StorageLevel.MEMORY_AND_DISK)
testDF  = testDF.persist(StorageLevel.MEMORY_AND_DISK)

print("Train rows:", trainDF.count(), "| Test rows:", testDF.count())

# Count class distribution
counts = {r["label"]: r["count"] for r in trainDF.groupBy("label").count().collect()}
ben_n = int(counts.get(0.0, 0)); att_n = int(counts.get(1.0, 0))

ben = trainDF.filter(F.col("label") == 0.0)
att = trainDF.filter(F.col("label") == 1.0)

# Undersampling the majority class
if ben_n > att_n:
    frac = att_n / ben_n if ben_n > 0 else 0.0
    ben_s = ben.sample(False, max(min(frac, 1.0), 0.0), seed=42)
    train_bal = ben_s.unionByName(att)
    sampled_major = "benign"; sample_ratio = frac
else:
    frac = ben_n / att_n if att_n > 0 else 0.0
    att_s = att.sample(False, max(min(frac, 1.0), 0.0), seed=42)
    train_bal = ben.unionByName(att_s)
    sampled_major = "attack"; sample_ratio = frac

train_bal = train_bal.repartition(target_parts).persist(StorageLevel.MEMORY_AND_DISK)
print("Balanced train rows:", train_bal.count(),
      "| benign:", ben_n, "| attack:", att_n,
      "| sampled_major:", sampled_major, "| sample_ratio:", round(sample_ratio, 4))


Train rows: 2516320 | Test rows: 629404
Balanced train rows: 1572248 | benign: 1730615 | attack: 785705 | sampled_major: benign | sample_ratio: 0.454


## Random Forest and Gradient Boosted Trees Training (4 Cores)

In [6]:
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time

# Number of partitions aligned with cores
parts = int(spark.conf.get("spark.sql.shuffle.partitions"))
train_bal = train_bal.repartition(parts).cache()
testDF    = testDF.repartition(parts).cache()

print("train_bal partitions:", train_bal.rdd.getNumPartitions(), "| testDF partitions:", testDF.rdd.getNumPartitions())

# Evaluators
e_bin = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
e_mc  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

# --- Random Forest (fast settings consistent with 8c) ---
rf = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=160, maxDepth=12, maxBins=64,
    subsamplingRate=0.8, featureSubsetStrategy="sqrt",
    cacheNodeIds=True, seed=42
)
t0 = time.time(); rf_m = rf.fit(train_bal); t1 = time.time()
rf_pred = rf_m.transform(testDF); t2 = time.time()

rf_auc = e_bin.evaluate(rf_pred)
rf_acc = e_mc.evaluate(rf_pred, {e_mc.metricName: "accuracy"})
rf_f1  = e_mc.evaluate(rf_pred, {e_mc.metricName: "f1"})
rf_fit_s, rf_pred_s = t1 - t0, t2 - t1
print(f"[RF]  AUC={rf_auc:.6f} | Acc={rf_acc:.6f} | F1={rf_f1:.6f} | fit_s={rf_fit_s:.1f} | pred_s={rf_pred_s:.2f}")

# --- GBT (reduce time while maintaining accuracy) ---
gbt = GBTClassifier(
    featuresCol="features", labelCol="label",
    maxIter=80, maxDepth=8, stepSize=0.15,
    maxBins=64,
    cacheNodeIds=False,   # reduce memory/time pressure
    seed=42
)
t0 = time.time(); gbt_m = gbt.fit(train_bal); t1 = time.time()
gbt_pred = gbt_m.transform(testDF); t2 = time.time()

gbt_auc = e_bin.evaluate(gbt_pred)
gbt_acc = e_mc.evaluate(gbt_pred, {e_mc.metricName: "accuracy"})
gbt_f1  = e_mc.evaluate(gbt_pred, {e_mc.metricName: "f1"})
gbt_fit_s, gbt_pred_s = t1 - t0, t2 - t1
print(f"[GBT] AUC={gbt_auc:.6f} | Acc={gbt_acc:.6f} | F1={gbt_f1:.6f} | fit_s={gbt_fit_s:.1f} | pred_s={gbt_pred_s:.2f}")


train_bal partitions: 24 | testDF partitions: 24
[RF]  AUC=0.999999 | Acc=0.999655 | F1=0.999655 | fit_s=284.7 | pred_s=0.15
[GBT] AUC=0.999995 | Acc=0.999930 | F1=0.999930 | fit_s=458.4 | pred_s=0.17


## Threshold Tuning with Constraints and Confusion Matrix Evaluation

In [7]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
import time

def ensure_score_column(pred_df):
    if "probability" in pred_df.columns:
        return pred_df.withColumn("p1", vector_to_array(F.col("probability"))[1])
    elif "rawPrediction" in pred_df.columns:
        rp1 = vector_to_array(F.col("rawPrediction"))[1]
        return pred_df.withColumn("p1", 1 / (1 + F.exp(-rp1)))
    else:
        raise ValueError("Missing 'probability' or 'rawPrediction'.")

def confusion_metrics_df(df_with_pred):
    agg = df_with_pred.agg(
        F.sum(F.when((F.col("label")==0) & (F.col("pred")==0), 1).otherwise(0)).alias("tn"),
        F.sum(F.when((F.col("label")==0) & (F.col("pred")==1), 1).otherwise(0)).alias("fp"),
        F.sum(F.when((F.col("label")==1) & (F.col("pred")==0), 1).otherwise(0)).alias("fn"),
        F.sum(F.when((F.col("label")==1) & (F.col("pred")==1), 1).otherwise(0)).alias("tp")
    ).collect()[0]
    tn, fp, fn, tp = float(agg["tn"]), float(agg["fp"]), float(agg["fn"]), float(agg["tp"])
    precision  = tp / (tp + fp + 1e-12)
    recall     = tp / (tp + fn + 1e-12)
    specificity= tn / (tn + fp + 1e-12)
    f1         = 2 * precision * recall / (precision + recall + 1e-12)
    fpr        = fp / (fp + tn + 1e-12)
    fnr        = fn / (fn + tp + 1e-12)
    return tn, fp, fn, tp, precision, recall, specificity, f1, fpr, fnr

def thresholds_from_quantiles(scored_df, n=31):
    qs = [i/(n-1) for i in range(n)]  # 0..1
    quants = scored_df.approxQuantile("p1", qs, 1e-4)
    uniq = sorted(set(float(x) for x in quants))
    uniq = [x for x in uniq if 0.05 <= x <= 0.95]
    uniq = sorted(set(uniq + [0.5]))   # include 0.5 as reference
    if len(uniq) < 7:
        uniq = sorted(set(uniq + [0.1, 0.2, 0.3, 0.7, 0.8, 0.9]))
    return uniq

def tune_threshold_safe(pred_df, name="Model", min_precision=0.99, min_recall=0.95, min_tp=100):
    parts = int(spark.conf.get("spark.sql.shuffle.partitions"))
    scored = ensure_score_column(pred_df).select("p1", "label").repartition(parts).cache()
    _ = scored.count()

    ths = thresholds_from_quantiles(scored, n=31)

    best = None
    best_key = -1.0  # maximize F1
    t0 = time.time()
    for th in ths:
        pr = scored.withColumn("pred", (F.col("p1") >= F.lit(th)).cast("double"))
        tn, fp, fn, tp, prec, rec, spec, f1, _, _ = confusion_metrics_df(pr)
        if (prec >= min_precision) and (rec >= min_recall) and (tp >= min_tp):
            if f1 > best_key:
                best_key = f1
                best = (th, prec, rec, spec, f1)

    # If constraints not satisfied, take best overall F1
    if best is None:
        for th in ths:
            pr = scored.withColumn("pred", (F.col("p1") >= F.lit(th)).cast("double"))
            _, _, _, _, prec, rec, spec, f1, _, _ = confusion_metrics_df(pr)
            if f1 > best_key:
                best_key = f1
                best = (th, prec, rec, spec, f1)

    t1 = time.time()
    th, prec, rec, spec, f1 = best
    print(f"[{name}] best threshold={th:.4f} | Precision={prec:.3f} | Recall={rec:.3f} | "
          f"Specificity={spec:.3f} | F1={f1:.3f} | tune_s={t1 - t0:.2f}")
    return best

def apply_threshold(pred_df, th):
    scored = ensure_score_column(pred_df)
    return scored.withColumn("prediction", (F.col("p1") >= F.lit(float(th))).cast("double"))

# Run tuning and reporting
best_rf  = tune_threshold_safe(rf_pred,  "RF")
best_gbt = tune_threshold_safe(gbt_pred, "GBT")

for name, pred_df, best in [
    ("RF_tuned",  rf_pred,  best_rf),
    ("GBT_tuned", gbt_pred, best_gbt),
]:
    tuned = apply_threshold(pred_df, best[0]).select("label","prediction").withColumnRenamed("prediction","pred")
    tn, fp, fn, tp, prec, rec, spec, f1, fpr, fnr = confusion_metrics_df(tuned)
    print(f"[{name}] CM: [[TN={tn:.0f} FP={fp:.0f}] [FN={fn:.0f} TP={tp:.0f}]] "
          f"| Precision={prec:.3f} | Recall={rec:.3f} | Specificity={spec:.3f} | F1={f1:.3f} "
          f"| FPR={fpr:.6f} | FNR={fnr:.6f}")


[RF] best threshold=0.8000 | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000 | tune_s=4.50
[GBT] best threshold=0.5000 | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000 | tune_s=3.60
[RF_tuned] CM: [[TN=431993 FP=3] [FN=49 TP=197359]] | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000 | FPR=0.000007 | FNR=0.000248
[GBT_tuned] CM: [[TN=431993 FP=3] [FN=41 TP=197367]] | Precision=1.000 | Recall=1.000 | Specificity=1.000 | F1=1.000 | FPR=0.000007 | FNR=0.000208


## Export Evaluation Metrics to CSV (4-Core Run)

In [8]:
import time
from pathlib import Path
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Default threshold metrics (0.5)
e_bin = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
e_mc  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

rf_auc_def  = e_bin.evaluate(rf_pred)
rf_acc_def  = e_mc.evaluate(rf_pred, {e_mc.metricName: "accuracy"})
rf_f1_def   = e_mc.evaluate(rf_pred, {e_mc.metricName: "f1"})

gbt_auc_def = e_bin.evaluate(gbt_pred)
gbt_acc_def = e_mc.evaluate(gbt_pred, {e_mc.metricName: "accuracy"})
gbt_f1_def  = e_mc.evaluate(gbt_pred, {e_mc.metricName: "f1"})

# Apply tuned thresholds
th_rf, th_gbt = best_rf[0], best_gbt[0]
rf_tuned  = apply_threshold(rf_pred,  th_rf)
gbt_tuned = apply_threshold(gbt_pred, th_gbt)

def collect_metrics(df, name, threshold):
    row = df.select("label","prediction").withColumnRenamed("prediction","pred").agg(
        F.sum(F.when((F.col("label")==0) & (F.col("pred")==0), 1).otherwise(0)).alias("tn"),
        F.sum(F.when((F.col("label")==0) & (F.col("pred")==1), 1).otherwise(0)).alias("fp"),
        F.sum(F.when((F.col("label")==1) & (F.col("pred")==0), 1).otherwise(0)).alias("fn"),
        F.sum(F.when((F.col("label")==1) & (F.col("pred")==1), 1).otherwise(0)).alias("tp")
    ).collect()[0]
    tn, fp, fn, tp = int(row["tn"]), int(row["fp"]), int(row["fn"]), int(row["tp"])
    prec = tp / (tp + fp + 1e-12); rec = tp / (tp + fn + 1e-12)
    f1   = 2*prec*rec / (prec + rec + 1e-12)
    fpr  = fp / (fp + tn + 1e-12); fnr = fn / (fn + tp + 1e-12)
    return {"Algorithm": name, "Threshold": round(float(threshold),4),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp,
            "Precision": prec, "Recall": rec, "F1": f1, "FPR": fpr, "FNR": fnr}

rows = [
    {**collect_metrics(rf_tuned,  "Random Forest",         th_rf),
     "AUC_default": rf_auc_def,  "Accuracy_default": rf_acc_def,  "F1_default": rf_f1_def,
     "fit_s": rf_fit_s, "pred_s": rf_pred_s},
    {**collect_metrics(gbt_tuned, "Gradient Boosted Tree", th_gbt),
     "AUC_default": gbt_auc_def, "Accuracy_default": gbt_acc_def, "F1_default": gbt_f1_def,
     "fit_s": gbt_fit_s, "pred_s": gbt_pred_s},
]

conf = spark.sparkContext.getConf()
rows = [{**r,
         "Master": spark.sparkContext.master,
         "AppName": spark.sparkContext.appName,
         "ExecutorCores": conf.get("spark.executor.cores"),
         "ExecutorInstances": conf.get("spark.executor.instances"),
         "TotalCores(tag)": 4,
         "DefaultParallelism": spark.sparkContext.defaultParallelism,
         "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        } for r in rows]

REPORTS = Path(r"C:\Users\HP\bigdata\sparkk2\reports"); REPORTS.mkdir(parents=True, exist_ok=True)
metrics_path = REPORTS / "worker_run_metrics_4c_opt.csv"  

import pandas as pd
df_metrics = pd.DataFrame(rows)
df_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
print("Saved metrics CSV ->", metrics_path)
display(df_metrics)


Saved metrics CSV -> C:\Users\HP\bigdata\sparkk2\reports\worker_run_metrics_4c_opt.csv


,Algorithm,Threshold,TN,FP,FN,TP,Precision,Recall,F1,FPR,...,F1_default,fit_s,pred_s,Master,AppName,ExecutorCores,ExecutorInstances,TotalCores(tag),DefaultParallelism,Timestamp
0,Random Forest,0.8,431993,3,49,197359,0.999985,0.999752,0.999868,0.000007,...,0.999655,284.686439,0.149352,spark://127.0.0.1:7077,IDS2018-Notebook-4c-1e,4,1,4,12,2025-09-25 17:31:35
1,Gradient Boosted Tree,0.5,431993,3,41,197367,0.999985,0.999792,0.999889,0.000007,...,0.999930,458.413522,0.165756,spark://127.0.0.1:7077,IDS2018-Notebook-4c-1e,4,1,4,12,2025-09-25 17:31:35
